In [ ]:
#%% Imports and Configuration
# =========================================================================================
# CSIRO BIOMASS: "GOLD STANDARD" HYBRID PIPELINE
# Strategy: DINOv2 + SigLIP (Split Stream) + Metadata -> CatBoost
# =========================================================================================

import os
import gc
import sys
import numpy as np
import pandas as pd
import cv2
from PIL import Image
from tqdm.auto import tqdm
from glob import glob
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import torch
import torchvision.transforms as T
import timm
from catboost import CatBoostRegressor, Pool
import warnings

warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', message='.*repr.*Field.*')
warnings.filterwarnings('ignore', message='.*frozen.*Field.*')

class CFG:
    SEED = 42
    N_FOLDS = 5
    DINO_IMG_SIZE = 518  # DINOv2 requires 518x518
    SIGLIP_IMG_SIZE = 384  # SigLIP requires 384x384
    BATCH_SIZE = 16 # Adjust based on GPU memory
    
    # PATHS - Kaggle uses /kaggle/input/<dataset-name>/
    # You need to add the dataset in Kaggle notebook settings
    ROOT = '/kaggle/input/csiro-biomass/'
    TRAIN_DIR = os.path.join(ROOT, 'train')
    TEST_DIR = os.path.join(ROOT, 'test')
    
    # MODELS
    # Using Large instead of Giant to ensure it fits in 16GB P100 memory
    # If you have A100, switch to 'vit_giant_patch14_dinov2.lvd142m'
    DINO_MODEL = 'vit_large_patch14_dinov2.lvd142m' 
    SIGLIP_MODEL = 'vit_so400m_patch14_siglip_384'
    
    TARGETS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']

def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

seed_everything(CFG.SEED)

# Verify Kaggle environment and paths
print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    
# Check if dataset is mounted
print(f"\nChecking data paths...")
print(f"ROOT exists: {os.path.exists(CFG.ROOT)}")
if os.path.exists(CFG.ROOT):
    print(f"ROOT contents: {os.listdir(CFG.ROOT)}")
print(f"TRAIN_DIR exists: {os.path.exists(CFG.TRAIN_DIR)}")
print(f"TEST_DIR exists: {os.path.exists(CFG.TEST_DIR)}")

In [ ]:
#%% 1. Metadata Engineering
def process_metadata(df):
    """
    Cleans and enhances tabular data.
    Based on actual CSIRO competition columns:
    - Sampling_Date, State, Species, Pre_GSHH_NDVI, Height_Ave_cm
    """
    df = df.copy()
    
    # Date Features (actual column name is 'Sampling_Date')
    if 'Sampling_Date' in df.columns:
        df['Sampling_Date'] = pd.to_datetime(df['Sampling_Date'], errors='coerce')
        # Fill missing dates if any (forward fill)
        df['Sampling_Date'] = df['Sampling_Date'].ffill()
        
        df['Month'] = df['Sampling_Date'].dt.month
        df['DayOfYear'] = df['Sampling_Date'].dt.dayofyear
        # Cyclic encoding for Month (Seasonality is key)
        df['Month_sin'] = np.sin(2 * np.pi * df['Month']/12)
        df['Month_cos'] = np.cos(2 * np.pi * df['Month']/12)
        df = df.drop(columns=['Sampling_Date', 'Month'])

    # One-Hot Encoding for Categoricals (actual columns: State, Species)
    categorical_cols = ['Species', 'State']
    for col in categorical_cols:
        if col in df.columns:
            dummies = pd.get_dummies(df[col], prefix=col, dummy_na=False)
            df = pd.concat([df, dummies], axis=1)
            df = df.drop(columns=[col])
            
    # Interaction Features
    # Pre_GSHH_NDVI * Height is the strongest physical predictor of biomass volume
    if 'Pre_GSHH_NDVI' in df.columns and 'Height_Ave_cm' in df.columns:
        df['Volume_Proxy'] = df['Pre_GSHH_NDVI'] * df['Height_Ave_cm']
        
    return df

print("Metadata processing function defined.")

In [ ]:
#%% 2. Global Color Stats
def get_color_stats(image_path):
    """
    Calculates explicit color descriptors (RGB Mean/Std/ExcessGreen).
    Fast and robust proxy for 'Greenness' vs 'Brownness'.
    """
    img = cv2.imread(image_path)
    if img is None: return np.zeros(9)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) / 255.0
    
    # Means and Stds
    means = img.mean(axis=(0,1))
    stds = img.std(axis=(0,1))
    
    # Excess Green Index (2G - R - B)
    ExG = 2*means[1] - means[0] - means[2]
    
    # Green-Red Ratio (Proxy for senescence)
    GR_Ratio = means[1] / (means[0] + 1e-6)
    
    # Return vector: [R_m, G_m, B_m, R_s, G_s, B_s, ExG, GR, 0]
    return np.concatenate([means, stds, [ExG, GR_Ratio, 0]])

print("Color stats function defined.")

In [ ]:
#%% 3. Feature Extractor (The Engine)
class HybridExtractor:
    def __init__(self):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        print(f"Loading {CFG.DINO_MODEL}...")
        self.dino = timm.create_model(CFG.DINO_MODEL, pretrained=True, num_classes=0)
        self.dino = self.dino.to(self.device).eval()
        
        print(f"Loading {CFG.SIGLIP_MODEL}...")
        self.siglip = timm.create_model(CFG.SIGLIP_MODEL, pretrained=True, num_classes=0)
        self.siglip = self.siglip.to(self.device).eval()
        
        # Separate transforms for each model
        self.dino_transform = T.Compose([
            T.Resize((CFG.DINO_IMG_SIZE, CFG.DINO_IMG_SIZE)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
        self.siglip_transform = T.Compose([
            T.Resize((CFG.SIGLIP_IMG_SIZE, CFG.SIGLIP_IMG_SIZE)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    @torch.no_grad()
    def extract_batch(self, image_paths):
        """
        Splits images into Left/Right halves and extracts concatenated features.
        Uses correct input sizes for each model (DINO: 518, SigLIP: 384).
        Output Dim: (Batch, DINO_Dim*2 + SigLIP_Dim*2)
        """
        dino_batch_tensors = []
        siglip_batch_tensors = []
        
        for p in image_paths:
            img = Image.open(p).convert('RGB')
            w, h = img.size
            
            # Split Left and Right halves
            img_left = img.crop((0, 0, w//2, h))
            img_right = img.crop((w//2, 0, w, h))
            
            # Apply model-specific transforms
            dino_batch_tensors.append(self.dino_transform(img_left))
            dino_batch_tensors.append(self.dino_transform(img_right))
            siglip_batch_tensors.append(self.siglip_transform(img_left))
            siglip_batch_tensors.append(self.siglip_transform(img_right))
            
        # Stack: [Left1, Right1, Left2, Right2, ...]
        dino_batch_input = torch.stack(dino_batch_tensors).to(self.device)
        siglip_batch_input = torch.stack(siglip_batch_tensors).to(self.device)
        
        # Inference
        dino_feats = self.dino(dino_batch_input).cpu().numpy()
        siglip_feats = self.siglip(siglip_batch_input).cpu().numpy()
        
        # Concatenate models: [Batch*2, DINO+SigLIP]
        combined = np.hstack([dino_feats, siglip_feats])
        
        # Reshape to merge Left/Right for each image
        # Final shape: [Batch, (DINO+SigLIP)*2]
        return combined.reshape(len(image_paths), -1)

print("HybridExtractor class defined.")

In [ ]:
#%% 4. Load Data and Process Metadata
# Discover actual dataset path on Kaggle (auto-detect)
if not os.path.exists(CFG.ROOT):
    print(f"⚠️ Default path {CFG.ROOT} not found. Searching for dataset...")
    # Search in /kaggle/input/ for the dataset
    input_dir = '/kaggle/input/'
    if os.path.exists(input_dir):
        subdirs = [d for d in os.listdir(input_dir) if os.path.isdir(os.path.join(input_dir, d))]
        print(f"Available datasets: {subdirs}")
        # Look for the CSIRO dataset (try common names)
        for subdir in subdirs:
            test_path = os.path.join(input_dir, subdir)
            if os.path.exists(os.path.join(test_path, 'train.csv')):
                CFG.ROOT = test_path + '/'
                CFG.TRAIN_DIR = os.path.join(CFG.ROOT, 'train')
                CFG.TEST_DIR = os.path.join(CFG.ROOT, 'test')
                print(f"✓ Found dataset at: {CFG.ROOT}")
                break

# Load Data
train_csv_path = os.path.join(CFG.ROOT, 'train.csv')
test_csv_path = os.path.join(CFG.ROOT, 'test.csv')

if not os.path.exists(train_csv_path):
    raise FileNotFoundError(f"train.csv not found at {train_csv_path}\nPlease add the CSIRO Biomass dataset to your Kaggle notebook.")
    
train_df_raw = pd.read_csv(train_csv_path)
test_df_raw = pd.read_csv(test_csv_path)

print(f"Train shape: {train_df_raw.shape}")
print(f"Train columns: {train_df_raw.columns.tolist()}")
print(f"\nTest shape: {test_df_raw.shape}")
print(f"Test columns: {test_df_raw.columns.tolist()}")

# The train data has one row per (image, target_name) pair
# We need to pivot it to have one row per image with all targets as columns
print("\nPivoting train data to wide format...")

# Get non-target columns (metadata columns)
id_cols = [c for c in train_df_raw.columns if c not in ['target_name', 'target']]

train_pivot = train_df_raw.pivot_table(
    index=id_cols,
    columns='target_name',
    values='target',
    aggfunc='first'
).reset_index()

# Flatten column names after pivot
train_pivot.columns.name = None
print(f"Train pivot shape: {train_pivot.shape}")

# For test, get unique images (test has one row per sample_id which is image+target combo)
# We need unique images for feature extraction
test_unique = test_df_raw.drop_duplicates(subset='image_path').copy()
print(f"Test unique images: {len(test_unique)}")

# Prepare Metadata
print("\nProcessing Metadata...")
train_df = process_metadata(train_pivot)
test_df = process_metadata(test_unique)

# Align Columns (Ensure Train/Test have same dummy columns)
# Exclude target columns, image_path, sample_id, target_name, target, and ID
exclude_cols = set(CFG.TARGETS + ['image_path', 'sample_id', 'ID', 'target_name', 'target'])
all_cols = list(set(train_df.columns) | set(test_df.columns))
meta_cols = [c for c in all_cols if c not in exclude_cols]

# Fill missing columns in both with 0
for c in meta_cols:
    if c not in train_df.columns: train_df[c] = 0
    if c not in test_df.columns: test_df[c] = 0

# Sort meta_cols for consistency
meta_cols = sorted(meta_cols)

print(f"\nTrain shape: {train_df.shape}, Test shape: {test_df.shape}")
print(f"Metadata columns ({len(meta_cols)}): {meta_cols[:10]}...")

In [ ]:
#%% 4b. DIAGNOSTIC - Check Actual Directory Structure
print("\n" + "="*70)
print("DIRECTORY STRUCTURE DIAGNOSTIC")
print("="*70)

# Check what's actually in the root directory
print(f"\n1. ROOT directory: {CFG.ROOT}")
if os.path.exists(CFG.ROOT):
    root_contents = os.listdir(CFG.ROOT)
    print(f"   Contents: {root_contents}")
    
    # Check each subdirectory
    for item in root_contents:
        item_path = os.path.join(CFG.ROOT, item)
        if os.path.isdir(item_path):
            print(f"\n   📁 {item}/")
            try:
                sub_contents = os.listdir(item_path)[:10]  # First 10 items
                print(f"      Contents (first 10): {sub_contents}")
            except:
                print(f"      (cannot list)")
else:
    print(f"   ❌ Does not exist!")

# Check TRAIN_DIR
print(f"\n2. TRAIN_DIR: {CFG.TRAIN_DIR}")
print(f"   Exists: {os.path.exists(CFG.TRAIN_DIR)}")
if os.path.exists(CFG.TRAIN_DIR):
    train_contents = os.listdir(CFG.TRAIN_DIR)[:5]
    print(f"   Contents (first 5): {train_contents}")

# Check TEST_DIR  
print(f"\n3. TEST_DIR: {CFG.TEST_DIR}")
print(f"   Exists: {os.path.exists(CFG.TEST_DIR)}")
if os.path.exists(CFG.TEST_DIR):
    test_contents = os.listdir(CFG.TEST_DIR)[:5]
    print(f"   Contents (first 5): {test_contents}")

# Check image paths from CSV
print(f"\n4. Sample image paths from train.csv:")
if 'train_df_raw' in globals():
    sample_paths = train_df_raw['image_path'].head(5).tolist()
    for i, path in enumerate(sample_paths):
        print(f"   {i+1}. {path}")
        # Try to find it
        check_paths = [
            os.path.join(CFG.ROOT, path),
            os.path.join(CFG.TRAIN_DIR, path),
            os.path.join(CFG.TRAIN_DIR, os.path.basename(path)),
        ]
        for cp in check_paths:
            if os.path.exists(cp):
                print(f"      ✓ FOUND at: {cp}")
                break
        else:
            print(f"      ❌ NOT FOUND in any expected location")

print("\n" + "="*70)

In [ ]:
#%% 5. Extract Features
# Initialize Extractor
extractor = HybridExtractor()

def get_full_features(df, img_dir):
    """
    Get full features for images. Handles Kaggle directory structure.
    CSV paths are like: 'train/ID1011485656.jpg' or 'test/ID1001187975.jpg'
    """
    image_paths = []
    valid_indices = []
    
    print(f"\nProcessing {len(df)} images from {img_dir}")
    
    for idx, img_path in enumerate(df['image_path']):
        # CSV path format: 'train/ID1011485656.jpg'
        # We need: '/kaggle/input/csiro-biomass/train/ID1011485656.jpg'
        
        # Try multiple strategies to find the image
        possible_paths = [
            os.path.join(CFG.ROOT, img_path),  # Most likely: ROOT + 'train/ID.jpg'
            os.path.join(img_dir, os.path.basename(img_path)),  # TRAIN_DIR + 'ID.jpg'
        ]
        
        full_path = None
        for path in possible_paths:
            if os.path.exists(path):
                full_path = path
                break
        
        if full_path is None:
            if idx < 3:  # Only print first few warnings
                print(f"  ⚠️ Warning: Image not found: {img_path}")
            continue
        
        image_paths.append(full_path)
        valid_indices.append(idx)
    
    print(f"✓ Found {len(image_paths)}/{len(df)} images")
    
    if len(image_paths) == 0:
        raise ValueError(
            f"❌ No images found!\n"
            f"Checked directories:\n"
            f"  - {CFG.ROOT}\n"
            f"  - {img_dir}\n"
            f"Sample CSV path: {df['image_path'].iloc[0] if len(df) > 0 else 'N/A'}"
        )
    
    # 1. Visual Features (DINOv2 + SigLIP)
    print(f"\nExtracting Visual Features from {len(image_paths)} images...")
    visual_feats = []
    
    for i in tqdm(range(0, len(image_paths), CFG.BATCH_SIZE), desc="Vision Models"):
        batch_paths = image_paths[i : i + CFG.BATCH_SIZE]
        try:
            visual_feats.append(extractor.extract_batch(batch_paths))
        except Exception as e:
            print(f"\n  ⚠️ Error processing batch {i//CFG.BATCH_SIZE + 1}: {e}")
            raise
    
    visual_feats = np.vstack(visual_feats)
    print(f"  → Visual features shape: {visual_feats.shape}")
    
    # 2. Color Statistics
    print(f"\nExtracting Color Statistics...")
    color_feats = []
    for p in tqdm(image_paths, desc="Color Stats"):
        color_feats.append(get_color_stats(p))
    color_feats = np.array(color_feats)
    print(f"  → Color features shape: {color_feats.shape}")
    
    # 3. Metadata Features
    print(f"\nProcessing Metadata Features...")
    meta_feats = df[meta_cols].iloc[valid_indices].values.astype(np.float32)
    meta_feats = np.nan_to_num(meta_feats)
    print(f"  → Metadata features shape: {meta_feats.shape}")
    
    # Combine All Features
    combined_feats = np.hstack([visual_feats, color_feats, meta_feats])
    print(f"\n✓ Total combined features shape: {combined_feats.shape}")
    
    return combined_feats, valid_indices

print("\n" + "="*70)
print("FEATURE EXTRACTION")
print("="*70)

print("\n>>> Processing TRAIN data <<<")
X, train_valid_idx = get_full_features(train_df, CFG.TRAIN_DIR)
Y = train_df[CFG.TARGETS].iloc[train_valid_idx].values

# Check for NaN in targets and remove them
print(f"\nValidating targets...")
nan_mask = np.isnan(Y).any(axis=1)
nan_count = nan_mask.sum()

if nan_count > 0:
    print(f"  ⚠️ Found {nan_count} rows with NaN targets. Removing them...")
    valid_mask = ~nan_mask
    X = X[valid_mask]
    Y = Y[valid_mask]
    print(f"  → After cleaning: X shape: {X.shape}, Y shape: {Y.shape}")
else:
    print(f"  ✓ No NaN values found")

print("\n>>> Processing TEST data <<<")
X_test, test_valid_idx = get_full_features(test_df, CFG.TEST_DIR)

# Keep track of test image paths for submission mapping
test_image_paths = test_df['image_path'].iloc[test_valid_idx].values

print(f"\n{'='*70}")
print(f"FINAL DATASET SHAPES")
print(f"{'='*70}")
print(f"  Training:   X={X.shape}, Y={Y.shape}")
print(f"  Test:       X_test={X_test.shape}")
print(f"  Features:   {X.shape[1]} total")
print(f"{'='*70}\n")

In [ ]:
#%% 5b. Verify Data Before Training
print(f"\n{'='*60}")
print("DATA VERIFICATION BEFORE TRAINING")
print(f"{'='*60}")
print(f"X shape: {X.shape}")
print(f"Y shape: {Y.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Number of samples: {len(X)}")
print(f"Number of features: {X.shape[1] if len(X) > 0 else 0}")
print(f"{'='*60}\n")

if len(X) == 0:
    print("❌ ERROR: No training samples found!")
    print("Possible causes:")
    print(f"1. Data path incorrect: {CFG.ROOT}")
    print(f"2. Train directory: {CFG.TRAIN_DIR}")
    print(f"3. Test directory: {CFG.TEST_DIR}")
    print("\nPlease update CFG.ROOT to point to your actual data location.")
    print("Example: CFG.ROOT = r'C:\\Users\\wadhw\\OneDrive\\Desktop\\csiro-biomass-data\\'")
    raise ValueError("No training data found. Update CFG.ROOT path and re-run.")

In [ ]:
#%% 6. Clear Memory
# Clear memory (keep train_df for later reference if needed)
del extractor
gc.collect()
torch.cuda.empty_cache()
print("Memory cleared.")

In [ ]:
#%% 7. Training (CatBoost)
print(">>> Starting Training <<<")

# Validate data before training
if len(X) == 0:
    raise ValueError(
        "❌ FATAL ERROR: No training data available!\n\n"
        "This means images were not found during feature extraction.\n"
        "SOLUTION:\n"
        "1. In Kaggle, click '+ Add Data' button\n"
        "2. Search for 'CSIRO Biomass' dataset\n"
        "3. Add it to your notebook\n"
        "4. Re-run all cells from the beginning\n\n"
        f"Current paths:\n"
        f"  ROOT: {CFG.ROOT}\n"
        f"  TRAIN_DIR: {CFG.TRAIN_DIR}\n"
        f"  TEST_DIR: {CFG.TEST_DIR}\n"
    )

print(f"✓ Training data loaded: {len(X)} samples, {X.shape[1]} features")
print(f"✓ Test data loaded: {len(X_test)} samples")

predictions = np.zeros((len(X_test), len(CFG.TARGETS)))
oof_preds = np.zeros((len(X), len(CFG.TARGETS)))

kf = KFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)

for i, target_name in enumerate(CFG.TARGETS):
    print(f"\n{'='*50}")
    print(f"Training Target [{i+1}/{len(CFG.TARGETS)}]: {target_name}")
    print(f"{'='*50}")
    y_target = Y[:, i]
    
    # Store predictions for this target
    target_preds = np.zeros(len(X_test))
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y_target)):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y_target[train_idx], y_target[val_idx]
        
        print(f"  Fold {fold+1}/{CFG.N_FOLDS} - Train: {len(X_tr)}, Val: {len(X_val)}", end=" ")
        
        model = CatBoostRegressor(
            iterations=2000,
            learning_rate=0.03,
            depth=6,
            l2_leaf_reg=5,
            loss_function='RMSE',
            eval_metric='R2',
            early_stopping_rounds=100,
            task_type='GPU', # Use GPU for training
            verbose=0,
            random_seed=CFG.SEED
        )
        
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val))
        
        # Predict
        val_pred = model.predict(X_val)
        val_pred = np.maximum(val_pred, 0) # Relu
        oof_preds[val_idx, i] = val_pred
        
        test_pred = model.predict(X_test)
        test_pred = np.maximum(test_pred, 0)
        target_preds += test_pred / CFG.N_FOLDS
        
        val_r2 = r2_score(y_val, val_pred)
        print(f"→ R2: {val_r2:.4f}")
        
    predictions[:, i] = target_preds
    score = r2_score(y_target, oof_preds[:, i])
    print(f"\n  ✓ {target_name} Overall OOF R2: {score:.4f}")

print(f"\n{'='*50}")
print(">>> Training Complete <<<")
print(f"{'='*50}")

In [ ]:
#%% 8. Post-Processing & Submission
# Constraint: Dry_Total_g should be sum of parts.
# The Total column is index 4, Parts are 0, 1, 2
# Weight based on which model we trust more (usually total is more accurate than parts)

# Simple correction: 50/50 Blend between predicted Total and Sum of components
sum_components = predictions[:, 0] + predictions[:, 1] + predictions[:, 2]
predictions[:, 4] = (predictions[:, 4] * 0.5) + (sum_components * 0.5)

# Formatting Submission
# test.csv format: each row has sample_id (e.g., "test/ID1001187975.jpg_Dry_Green_g")
# We need to map our predictions (per unique image) back to each sample_id

# Create prediction lookup: image_path -> {target_name: prediction}
pred_dict = {}
for i, img_path in enumerate(test_image_paths):
    pred_dict[img_path] = {}
    for j, target in enumerate(CFG.TARGETS):
        pred_dict[img_path][target] = float(predictions[i, j])

# Build submission from original test format
sub_rows = []
for _, row in test_df_raw.iterrows():
    img_path = row['image_path']
    target_name = row['target_name']
    sample_id = row['sample_id']
    
    # Look up prediction
    if img_path in pred_dict and target_name in pred_dict[img_path]:
        pred_value = pred_dict[img_path][target_name]
    else:
        pred_value = 0.0
        print(f"Warning: Missing prediction for {sample_id}")
    
    sub_rows.append({
        'sample_id': sample_id,
        'target': pred_value
    })

submission = pd.DataFrame(sub_rows)
submission.to_csv('submission.csv', index=False)
print("Submission Saved!")
print(f"Submission shape: {submission.shape}")
print(submission.head(10))